In [0]:

# Databricks notebook source
# MAGIC %md
# MAGIC # Silver: member dimension with SCD Type 2
# MAGIC Tracks the full history of member profile changes (for example, moving provinces) instead of overwriting them.
# MAGIC
# MAGIC - **Source:** `personal_finance_bronze.members_raw`, written by `synthetic_banking_generator`.
# MAGIC - **Change detection:** a hash of the tracked columns. Only a real change creates a new version.
# MAGIC - **One Delta MERGE per load:** expires the old version and inserts the new one in a single atomic operation.
# MAGIC - **Idempotent:** applying the same snapshot twice changes nothing. The validation cell proves it.
# MAGIC
# MAGIC Why it matters for TrueNorth: TFSA rules depend on facts like residency. With SCD2 you can answer "what did we know about this member on a given date?", which is what audits ask.
# MAGIC
# MAGIC This notebook rebuilds the table on each run so the demo is repeatable. A production pipeline would keep the table and only run `apply_scd2`.
 
# COMMAND ----------
 
from delta.tables import DeltaTable
from pyspark.sql.functions import col, concat_ws, lit, sha2, when
 
SOURCE = "personal_finance_bronze.members_raw"
TARGET = "personal_finance_silver.dim_member_scd2"
TRACKED = ["first_name", "last_name", "birth_year", "province", "resident_since"]
OPEN_END = "9999-12-31"
 
spark.sql("CREATE SCHEMA IF NOT EXISTS personal_finance_silver")
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")   # demo only, see note above
spark.sql(f"""
    CREATE TABLE {TARGET} (
        member_id            BIGINT,
        first_name           STRING,
        last_name            STRING,
        birth_year           INT,
        province             STRING,
        resident_since       INT,
        row_hash             STRING,
        effective_start_date DATE,
        effective_end_date   DATE,
        is_current           BOOLEAN
    ) USING DELTA
""")
 
# COMMAND ----------
 
def with_hash(df):
    """Add a hash of the tracked columns so a change is one comparison."""
    return df.withColumn(
        "row_hash", sha2(concat_ws("||", *[col(c).cast("string") for c in TRACKED]), 256)
    )
 
 
def apply_scd2(snapshot_df, as_of):
    """Merge a member snapshot into the SCD2 dimension as of a given date (YYYY-MM-DD)."""
    updates = with_hash(snapshot_df.select("member_id", *TRACKED))
    target = DeltaTable.forName(spark, TARGET)
    current = target.toDF().filter("is_current")
 
    # Members whose tracked attributes changed: they need a new version inserted
    changed = (
        updates.alias("u")
        .join(current.alias("t"), "member_id")
        .filter(col("u.row_hash") != col("t.row_hash"))
        .select("u.*")
    )
 
    # Classic staged-merge pattern:
    #  - rows keyed by member_id expire the current version when the hash differs
    #  - the same changed rows with a NULL key never match, so they insert the new version
    staged = (
        updates.withColumn("merge_key", col("member_id"))
        .unionByName(changed.withColumn("merge_key", lit(None).cast("bigint")))
    )
 
    (
        target.alias("t")
        .merge(staged.alias("s"), "t.member_id = s.merge_key AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={"is_current": "false", "effective_end_date": f"date_sub(DATE'{as_of}', 1)"},
        )
        .whenNotMatchedInsert(
            values={
                **{c: f"s.{c}" for c in ["member_id", *TRACKED, "row_hash"]},
                "effective_start_date": f"DATE'{as_of}'",
                "effective_end_date": f"DATE'{OPEN_END}'",
                "is_current": "true",
            }
        )
        .execute()
    )
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 1. Initial load
 
# COMMAND ----------
 
snapshot_jan = spark.table(SOURCE)
apply_scd2(snapshot_jan, "2026-01-01")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 2. A later snapshot where five members moved province
 
# COMMAND ----------
 
movers = [1000, 1001, 1002, 1003, 1004]
snapshot_nov = snapshot_jan.withColumn(
    "province",
    when(col("member_id").isin(movers) & (col("province") != "ON"), lit("ON"))
    .when(col("member_id").isin(movers), lit("BC"))
    .otherwise(col("province")),
)
apply_scd2(snapshot_nov, "2026-11-01")
 
rows_after_change = spark.table(TARGET).count()
 
# Apply the exact same snapshot again: an idempotent merge changes nothing
apply_scd2(snapshot_nov, "2026-11-01")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 3. Validation
 
# COMMAND ----------
 
dim = spark.table(TARGET)
num_members = snapshot_jan.count()
 
assert dim.count() == rows_after_change, "Re-applying the same snapshot added rows: merge is not idempotent"
assert dim.count() == num_members + len(movers), f"Expected {num_members + len(movers)} rows, got {dim.count()}"
 
multi_current = dim.filter("is_current").groupBy("member_id").count().filter("count > 1").count()
assert multi_current == 0, f"{multi_current} members have more than one current version"
 
assert dim.filter("is_current").count() == num_members, "Every member needs exactly one current version"
 
bad_ranges = dim.filter("effective_end_date < effective_start_date").count()
assert bad_ranges == 0, f"{bad_ranges} versions end before they start"
 

print(f"SCD2 dimension holds {dim.count():,} versions for {num_members:,} members. All checks passed.")
 